In [2]:
import anndata as ad
import pandas as pd
import numpy as np
import mofax as mfx
import duckdb as db

In [11]:


# --- RUTAS ---
MOFA_MODEL = '/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/modelos/modelo_final/modelo_mofa_30factors.hdf5'
INPUT_PARQUET = "/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/datos/datos_con_placa_14/tidy_final.parquet"
DRUG_PARQUET = '/mnt/lustre/scratch/nlsas/home/ulc/co/mao/drug.parquet'
OUTPUT_H5AD = '/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/datos/datos_con_placa_14/mofa_adata.h5ad'

# 1. Cargar modelo MOFA
print("Cargando modelo MOFA...")
model = mfx.mofa_model(MOFA_MODEL)

# 2. Factores (muestras × factores)
Z = model.get_factors(df=True)
print(f"Factores: {Z.shape}")
print(f"Índice ejemplo: {Z.index[:3].tolist()}")

# 3. Metadata desde el índice de los factores
print("Parseando metadata desde índice...")
parsed = Z.index.to_series().str.split('__', expand=True)
obs = pd.DataFrame({
    'drug': parsed[0].values,
    'concentration': parsed[1].values,
    'plate': parsed[2].values
}, index=Z.index)

# Limpiar drug
obs['drug'] = obs['drug'].str.replace(' ', '_').str.rstrip('_')

# 4. Añadir MOA desde drug metadata
print("Añadiendo MOA...")
drug_meta = pd.read_parquet(DRUG_PARQUET)
# Limpiar drug en drug_meta igual que en obs
drug_meta['drug'] = drug_meta['drug'].str.replace(' ', '_').str.rstrip('_')

# Merge MOA (ajusta nombre de columna si es diferente)
if 'moa-fine' in drug_meta.columns:
    moa_map = drug_meta[['drug', 'moa-fine']].drop_duplicates('drug')
    obs = obs.merge(moa_map, on='drug', how='left')
    obs.index = Z.index  # restaurar índice tras merge
    print(f"  MOA añadido. NaN: {obs['moa-fine'].isna().sum()}")
else:
    print(f"  Columnas disponibles en drug_meta: {drug_meta.columns.tolist()}")
    print("  Ajusta el nombre de la columna MOA manualmente")

# 5. Pesos de MOFA por vista (genes × factores)
print("Extrayendo pesos MOFA...")
W = model.get_weights(df=True)
weights = W
views = model.get_views()
for v in views:
    W.index = W.index.str.removesuffix(f'{v}')  # Limpiar sufijo de vista

# 6. Crear AnnData principal (muestras × factores)
print("Creando AnnData...")
adata = ad.AnnData(
    X=Z.values,
    obs=obs,
    var=pd.DataFrame(index=Z.columns)  # Factor1..Factor30
)

# 7. Guardar pesos en uns
adata.uns['mofa_weights'] = W.values
adata.uns['mofa_weights_genes'] = W.index.tolist()
adata.uns['mofa_weights_factors'] = W.columns.tolist()
adata.uns['mofa_views'] = list(model.views)

# Pesos por vista
for view, W in weights.items():
    adata.uns[f'mofa_weights_{view}'] = W.values

# 8. Guardar
print(f"\n{adata}")
print(f"\nobs columns: {adata.obs.columns.tolist()}")
adata.write(OUTPUT_H5AD, compression='gzip')
print(f"Guardado en {OUTPUT_H5AD}")

Cargando modelo MOFA...
Factores: (1297, 30)
Índice ejemplo: ['4EGI-1__5.0__3', '9-ING-41__5.0__3', 'APTO-253__5.0__3']
Parseando metadata desde índice...
Añadiendo MOA...
  MOA añadido. NaN: 0
Extrayendo pesos MOFA...
Creando AnnData...

AnnData object with n_obs × n_vars = 1297 × 30
    obs: 'drug', 'concentration', 'plate', 'moa-fine'
    uns: 'mofa_weights', 'mofa_weights_genes', 'mofa_weights_factors', 'mofa_views', 'mofa_weights_Factor1', 'mofa_weights_Factor2', 'mofa_weights_Factor3', 'mofa_weights_Factor4', 'mofa_weights_Factor5', 'mofa_weights_Factor6', 'mofa_weights_Factor7', 'mofa_weights_Factor8', 'mofa_weights_Factor9', 'mofa_weights_Factor10', 'mofa_weights_Factor11', 'mofa_weights_Factor12', 'mofa_weights_Factor13', 'mofa_weights_Factor14', 'mofa_weights_Factor15', 'mofa_weights_Factor16', 'mofa_weights_Factor17', 'mofa_weights_Factor18', 'mofa_weights_Factor19', 'mofa_weights_Factor20', 'mofa_weights_Factor21', 'mofa_weights_Factor22', 'mofa_weights_Factor23', 'mofa_wei

In [21]:
adata.X  # Ejemplo de acceso a pesos del factor 1

array([[-1.23099229, -1.71347897,  4.8358191 , ...,  0.79057662,
        -0.21964807, -0.83374637],
       [-5.78865116, -1.87250573,  4.50417212, ...,  0.07961434,
        -0.17502682, -0.41809349],
       [-2.14253081, -1.05068476,  3.90750094, ...,  0.34217753,
        -0.08704815, -0.34504714],
       ...,
       [ 2.89288681, -2.3001871 ,  0.49276723, ...,  0.48967143,
        -0.38869823,  0.82528126],
       [ 3.32043966,  0.09972039,  1.36675228, ...,  0.23040725,
        -0.11846083,  0.12387621],
       [ 2.34133355, -2.3633028 , -1.53623251, ...,  0.01613851,
        -0.12773025, -0.58676173]])

In [4]:
DRUG_PARQUET = '/mnt/lustre/scratch/nlsas/home/ulc/co/mao/drug.parquet'
drug_meta = pd.read_parquet(DRUG_PARQUET)

In [9]:

adata = ad.read_h5ad(OUTPUT_H5AD)

In [16]:
adata.obs

,drug,concentration,plate,moa-fine
4EGI-1_5.0__3__A549,4EGI-1_5.0,3,A549,NaN
9-ING-41_5.0__3__A549,9-ING-41_5.0,3,A549,NaN
APTO-253_5.0__3__A549,APTO-253_5.0,3,A549,NaN
AT7519_5.0__3__A549,AT7519_5.0,3,A549,NaN
AZD1390_5.0__3__A549,AZD1390_5.0,3,A549,NaN
...,...,...,...,...
TRIFLURIDINE_0.5__5__LOX-IMVI,TRIFLURIDINE_0.5,5,LOX-IMVI,NaN
TRIMETREXATE_0.5__5__LOX-IMVI,TRIMETREXATE_0.5,5,LOX-IMVI,NaN
TUCIDINOSTAT_0.5__5__LOX-IMVI,TUCIDINOSTAT_0.5,5,LOX-IMVI,NaN
VILANTEROL_0.5__5__LOX-IMVI,VILANTEROL_0.5,5,LOX-IMVI,NaN
